# 11 — Difference-in-Differences for Business Rollouts
**References:** Baker, Callaway, Cunningham, Goodman-Bacon & Sant'Anna (2026) "Difference-in-Differences Designs: A Practitioner's Guide" (*JEL* 64(2)) · Goodman-Bacon (2021, *J. Econometrics*) · Callaway & Sant'Anna (2021, *J. Econometrics*) · Card & Krueger (1994, *AER*) · Cunningham (2021) *The Mixtape* Ch. 9

**Prerequisites:** causal_inference_course/09_difference_in_differences.ipynb (full identification
theory, parallel trends, group-time ATT derivation — not repeated here).
**Connects to:** causal_inference_course/09_difference_in_differences.ipynb (the theory this
notebook applies); 15_event_study.ipynb (the diagnostic plot for the design below); 14_fixed_effects_panel.ipynb
(the within estimator underlying two-way fixed effects).

## Narrative thread
```
Business question: did the regional feature rollout work?
   -> Naive before/after and cross-region comparisons fail
   -> 2x2 DiD as the fix -> staggered rollout across regions/months
   -> TWFE danger with staggered timing (Goodman-Bacon decomposition, applied level)
   -> Callaway-Sant'Anna group-time ATT, applied level
   -> Practitioner decision checklist
```

## Why this notebook exists

Product and growth teams can't always run a clean A/B test: a pricing change, a new checkout
flow, or a loyalty program often rolls out **market by market** or **region by region** over
several months because of engineering constraints, regulatory approval, or a deliberate phased
launch. That staggered rollout is exactly the setting difference-in-differences (DiD) was built
for — and exactly the setting where the classic two-way fixed effects (TWFE) regression can
silently give the wrong answer. `causal_inference_course/09` derives the full theory (parallel
trends, event studies, Callaway & Sant'Anna's ATT(g,t)); this notebook applies it to a concrete
business rollout and focuses on **what to do**, not on re-deriving **why it works**.

## The business setting

Suppose a subscription product rolls out a **redesigned onboarding flow** to different
metro regions in different months (an engineering/ops constraint, not a designed experiment).
We observe monthly `signups_converted / signups_started` (activation rate) for 40 regions over
24 months. Regions "go live" with the new flow at different times; a few regions never get it
in the observation window (our comparison group).

This is a **staggered adoption** design: exactly the case Goodman-Bacon (2021) and Callaway &
Sant'Anna (2021) warn about.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(11)

In [2]:
# ── Simulate a staggered regional rollout of a new onboarding flow ──
n_regions = 40
n_months = 24
regions = np.arange(n_regions)
months = np.arange(n_months)

# region fixed effects (baseline activation rate heterogeneity across markets)
region_fe = np.random.normal(0, 0.03, n_regions)
# common month shocks (seasonality / macro trends shared by all regions -> satisfies parallel trends)
month_fe = 0.002 * months + np.random.normal(0, 0.01, n_months)

# adoption month: regions go live in waves; 8 regions never adopt in-window (never-treated)
adopt_month = np.random.choice(
    [6, 10, 14, 18, np.inf], size=n_regions, p=[0.2, 0.25, 0.25, 0.2, 0.1]
)

true_effect = 0.05  # true steady-state lift in activation rate from the new onboarding flow

rows = []
for r in range(n_regions):
    for t in range(n_months):
        treated = t >= adopt_month[r]
        base = 0.35 + region_fe[r] + month_fe[t]
        effect = true_effect if treated else 0.0
        activation = base + effect + np.random.normal(0, 0.015)
        rows.append((r, t, adopt_month[r], treated, activation))

panel = pd.DataFrame(rows, columns=["region", "month", "adopt_month", "treated", "activation"])
panel["cohort"] = panel["adopt_month"].replace(np.inf, -1).astype(int)
panel.head()

,region,month,adopt_month,treated,activation,cohort
0,0,0,6.0,False,0.375525,6
1,0,1,6.0,False,0.382451,6
2,0,2,6.0,False,0.417026,6
3,0,3,6.0,False,0.409954,6
4,0,4,6.0,False,0.420429,6


## Naive comparisons fail

Two tempting shortcuts and why each is wrong here:

- **Before/after within adopting regions only**: confounds the treatment effect with the
  month fixed effect (seasonality, product-wide trends) — any market-wide drift gets
  attributed to the rollout.
- **Treated vs. never-treated, pooled over all months**: confounds the treatment effect with
  time-invariant regional differences (region fixed effects) — some regions simply convert
  better regardless of the flow.

DiD nets out both by comparing the **change** in adopters against the **change** in a
comparison group over the same period.

In [3]:
# ── Naive before/after within ever-adopted regions (ignores month FE) ──
adopters = panel[panel["adopt_month"] != -1].copy()
before = adopters[adopters["month"] < adopters["adopt_month"]]["activation"].mean()
after = adopters[adopters["month"] >= adopters["adopt_month"]]["activation"].mean()
print(f"Naive before/after (adopters only): {after - before:.4f}  (true effect = {true_effect:.4f})")
print("-> biased upward/downward by whatever the shared month trend was doing over the same window.")

Naive before/after (adopters only): 0.0745  (true effect = 0.0500)
-> biased upward/downward by whatever the shared month trend was doing over the same window.


## Two-way fixed effects (TWFE) — and its staggered-timing danger

The standard regression is

$$Y_{it} = \alpha_i + \lambda_t + \tau\, D_{it} + \varepsilon_{it}$$

with region fixed effects $\alpha_i$, month fixed effects $\lambda_t$, and $D_{it}=1$ once
region $i$ has adopted. Goodman-Bacon (2021) shows that with **staggered adoption and
treatment-effect heterogeneity over time**, the TWFE coefficient $\hat\tau$ is a weighted
average of *all* possible 2x2 DiD comparisons — including comparisons of **already-treated**
regions against **later-treated** regions used as the "control." If the effect is not constant
(e.g., it builds up after go-live, as it typically does for a product change), those
already-treated-as-control comparisons can get **negative weight**, and the aggregate
$\hat\tau$ can be biased, or even have the wrong sign, even though every individual region
saw a positive effect. We do not re-derive the decomposition math here (see
`causal_inference_course/09`); the takeaway for practitioners is operational.

In [4]:
# ── TWFE regression on the full staggered panel ──
twfe = smf.ols("activation ~ treated + C(region) + C(month)", data=panel).fit(
    cov_type="cluster", cov_kwds={"groups": panel["region"]}
)
print(f"TWFE estimate of tau: {twfe.params['treated[T.True]']:.4f}  (true = {true_effect:.4f})")
print(f"Cluster-robust SE (by region): {twfe.bse['treated[T.True]']:.4f}")

TWFE estimate of tau: 0.0497  (true = 0.0500)
Cluster-robust SE (by region): 0.0020


In this simulation the effect is constant over time (no dynamic build-up), so TWFE happens to
recover something close to the truth. That is the trap: **TWFE can look fine on well-behaved
simulated data and still be dangerous on real rollouts**, because real product effects are
rarely constant — they ramp up as users adapt, or fade as novelty wears off (see
`causal_experimentation_course/01_ab_testing_fundamentals.ipynb` on novelty/primacy effects).
The Goodman-Bacon decomposition below quantifies *how much* of the TWFE estimate comes from
"forbidden" already-treated-vs-later-treated comparisons, at an applied level (no derivation).

In [5]:
# ── Goodman-Bacon decomposition, applied level: enumerate the 2x2 comparisons TWFE implicitly averages ──
# For each pair of cohorts (including the never-treated group as cohort -1), compute the
# simple 2x2 DiD and the implied Bacon weight (proportional to group-size product x
# variance of the treatment share within the comparison window). This reproduces the logic
# of Goodman-Bacon (2021) without the full variance-weighting proof.
cohorts = sorted(panel["cohort"].unique())
comparisons = []
for i, g1 in enumerate(cohorts):
    for g2 in cohorts[i + 1:]:
        # "early" cohort adopts first (or is never-treated, coded -1 -> treat as adopting at +inf)
        c1, c2 = (g1, g2) if (g1 != -1 and (g2 == -1 or g1 < g2)) else (g2, g1)
        c2_adopt = panel.loc[panel.cohort == c2, "adopt_month"].iloc[0]
        window_end = n_months if c2_adopt == np.inf else int(c2_adopt)  # stop before c2 itself adopts (if it ever does)
        sub = panel[panel["cohort"].isin([c1, c2]) & (panel.month < window_end)]
        split = int(sub.loc[sub.cohort == c1, "adopt_month"].iloc[0])  # c1's go-live splits pre/post for BOTH cohorts
        # 2x2 comparison window: c1's own before/after, minus c2's before/after over the same calendar window
        # (c2 has not yet adopted here by construction, so its "after" is a valid comparison-group trend)
        did = (
            sub[(sub.cohort == c1) & (sub.month >= split)]["activation"].mean()
            - sub[(sub.cohort == c1) & (sub.month < split)]["activation"].mean()
        ) - (
            sub[(sub.cohort == c2) & (sub.month >= split)]["activation"].mean()
            - sub[(sub.cohort == c2) & (sub.month < split)]["activation"].mean()
        )
        n1, n2 = (panel.cohort == c1).sum(), (panel.cohort == c2).sum()
        forbidden = (c1 != -1) and (c2 != -1)  # both cohorts eventually treated -> "treated-vs-treated" comparison
        comparisons.append((c1, c2, did, n1 * n2, forbidden))

bacon = pd.DataFrame(comparisons, columns=["cohort_1", "cohort_2", "diD_2x2", "weight_raw", "forbidden_comparison"])
bacon["weight_share"] = bacon["weight_raw"] / bacon["weight_raw"].sum()
share_forbidden = bacon.loc[bacon.forbidden_comparison, "weight_share"].sum()
print(bacon.drop(columns="weight_raw").round(4).to_string(index=False))
print(f"\nShare of TWFE weight coming from already-treated-vs-later-treated ('forbidden') comparisons: {share_forbidden:.1%}")

 cohort_1  cohort_2  diD_2x2  forbidden_comparison  weight_share
        6        -1   0.0527                 False        0.0166
       10        -1   0.0522                 False        0.0398
       14        -1   0.0498                 False        0.0365
       18        -1   0.0482                 False        0.0332
        6        10   0.0496                  True        0.0995
        6        14   0.0529                  True        0.0912
        6        18   0.0503                  True        0.0829
       10        14   0.0462                  True        0.2189
       10        18   0.0501                  True        0.1990
       14        18   0.0510                  True        0.1824

Share of TWFE weight coming from already-treated-vs-later-treated ('forbidden') comparisons: 87.4%


## Callaway & Sant'Anna (2021): group-time ATT, applied

The modern fix estimates one clean 2x2 DiD per (adoption cohort, calendar time) pair, using
**only not-yet-treated or never-treated units as the comparison group** — never an
already-treated unit. `causal_inference_course/09` derives $ATT(g,t)$ formally; here we
implement the applied workflow: for each cohort $g$ and each post-period $t$, compare the
change in outcome for cohort $g$ against the change for the not-yet-treated units, then
aggregate.

In [6]:
# ── Callaway-Sant'Anna style group-time ATT, applied implementation ──
def att_gt(panel, g, t, base_period=None):
    '''2x2 DiD for cohort g at time t, vs. units not-yet-treated by time t (or never-treated).'''
    if base_period is None:
        base_period = g - 1
    comparison = panel[(panel.cohort == -1) | (panel.cohort > t)]
    treated_grp = panel[panel.cohort == g]
    y_treat_t = treated_grp[treated_grp.month == t]["activation"].mean()
    y_treat_base = treated_grp[treated_grp.month == base_period]["activation"].mean()
    y_comp_t = comparison[comparison.month == t]["activation"].mean()
    y_comp_base = comparison[comparison.month == base_period]["activation"].mean()
    return (y_treat_t - y_treat_base) - (y_comp_t - y_comp_base)

adopt_cohorts = [c for c in cohorts if c != -1]
records = []
for g in adopt_cohorts:
    for t in range(g, n_months):
        records.append((g, t, t - g, att_gt(panel, g, t)))

cs = pd.DataFrame(records, columns=["cohort", "month", "event_time", "att_gt"])
# simple (unweighted) aggregation across cohorts and post periods, mirroring Callaway & Sant'Anna's overall ATT
overall_att = cs["att_gt"].mean()
print(f"Callaway-Sant'Anna style overall ATT (simple average of ATT(g,t)): {overall_att:.4f}  (true = {true_effect:.4f})")
print(cs.groupby("cohort")["att_gt"].mean().round(4))

Callaway-Sant'Anna style overall ATT (simple average of ATT(g,t)): 0.0455  (true = 0.0500)
cohort
6     0.0386
10    0.0577
14    0.0437
18    0.0408
Name: att_gt, dtype: float64


Notice the CS-style estimate never lets an already-treated cohort serve as the comparison
group for another cohort, so it is immune to the Goodman-Bacon "forbidden comparison" problem
by construction — at the cost of a more involved estimator and (in real packages, e.g. R's
`did` or Python's `csdid`/`differences`) proper multiplier-bootstrap standard errors, which we
do not implement from scratch here.

## Practitioner decision checklist for a staggered business rollout

1. **Is adoption staggered?** If every unit switches at the same calendar time, plain 2x2 DiD
   or standard TWFE is fine — skip the rest of this checklist.
2. **Is the treatment effect plausibly constant over time since adoption?** If yes (e.g., a
   one-time price change with an immediate, stable effect), TWFE bias is small. If no (ramp-up,
   novelty decay, seasonanl learning), TWFE is suspect — use Callaway & Sant'Anna, Sun &
   Abraham, or another heterogeneity-robust estimator.
3. **Do you have never-treated or late-treated units?** They are required as a clean
   comparison group; if literally everyone eventually adopts, use a not-yet-treated comparison
   group and be explicit about which periods are usable.
4. **Run the Goodman-Bacon decomposition** (or check the estimator's diagnostic) to see what
   share of the naive TWFE weight comes from already-treated-vs-later-treated comparisons —
   a large "forbidden" share is a red flag even before running the fix.
5. **Plot the event study** (`15_event_study.ipynb`) — pre-trends and the dynamic path of the
   effect are the single most informative diagnostic, and are required before trusting any
   DiD result in a business setting where "parallel trends" was never designed for, only hoped for.
6. **Report the estimator you used and why** — reviewers increasingly expect an explicit
   justification for TWFE vs. a heterogeneity-robust alternative (Baker et al. 2026).

## Key takeaways

| Concept | Statement |
|---|---|
| Staggered rollout | Common in business (phased launches); triggers the TWFE bias risk |
| TWFE bias mechanism | Implicitly compares already-treated vs. later-treated units, can get negative weight |
| Goodman-Bacon decomposition | Quantifies how much TWFE weight is "forbidden"; a diagnostic, not a fix |
| Callaway & Sant'Anna | Estimate ATT(g,t) using only not-yet/never-treated comparisons, then aggregate |
| Decision rule | Constant effect + no staggering concerns -> TWFE is fine; otherwise use a modern estimator |

## References

| Author(s) | Title | Role here |
|---|---|---|
| Goodman-Bacon (2021, *J. Econometrics*) | "Difference-in-Differences with Variation in Treatment Timing" | TWFE decomposition |
| Callaway & Sant'Anna (2021, *J. Econometrics*) | "Difference-in-Differences with Multiple Time Periods" | Group-time ATT estimator |
| Baker, Callaway, Cunningham, Goodman-Bacon & Sant'Anna (2026, *JEL*) | "Difference-in-Differences Designs: A Practitioner's Guide" | Practitioner checklist |
| Card & Krueger (1994, *AER*) | "Minimum Wages and Employment" | Canonical 2x2 DiD motivating example |
| Cunningham (2021) | *Causal Inference: The Mixtape*, Ch. 9 | Applied exposition |
